In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.analytic import ProbabilityOfImprovement
import copy

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_RBF.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 1.9540225650073515, 'n_it': 1.2447620531192634}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0, 1])),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": ProbabilityOfImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7):
        IterationClient = copy.deepcopy(client)
        IterationTrials = {}
        for __ in range(3):
            SampleTrial = IterationClient.get_next_trials(max_trials=1)
            for trial_index, parameters in SampleTrial.items():
                IterationTrials[trial_index]=parameters
                s1 = parameters["s1"]
                s2 = parameters["s2"]
                b1 = parameters["b1"]
                result = IterationClient.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
                raw_data = {metric_name: result}
                IterationClient.complete_trial(trial_index=trial_index, raw_data=raw_data)
        for trial_index, parameters in IterationTrials.items():
            client.attach_trial(parameters=parameters)
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(np.array(client.summarize().t1).tolist()[0:27]))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.718780135141754

Trial 1 =========================================
15.140761744346028

Trial 2 =========================================
15.115421216308118

Trial 3 =========================================
14.66197904907688

Trial 4 =========================================
15.143119125830182

Trial 5 =========================================
15.128998035991236

Trial 6 =========================================
15.141162558666206

Trial 7 =========================================
15.0951820894145

Trial 8 =========================================
15.1228548249069

Trial 9 =========================================
15.142239516127237

Trial 10 =========================================
15.140542559265285

Trial 11 =========================================
15.14049544868148

Trial 12 =========================================
15.142755956743358

Trial 13 =========================================
15.143534109723822

Trial 14 =============

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 24 =========================================
15.127779455441807

Trial 25 =========================================
15.119938754298055

Trial 26 =========================================
15.143550903716138

Trial 27 =========================================
14.903334712701191

Trial 28 =========================================
15.143282161723091

Trial 29 =========================================
15.143145666202487

Trial 30 =========================================
15.137713342893727

Trial 31 =========================================
15.04037609956161

Trial 32 =========================================
15.13860060888496

Trial 33 =========================================
15.093387013010044

Trial 34 =========================================
15.14248177460907

Trial 35 =========================================
15.057953924987444

Trial 36 =========================================
15.140282624068348

Trial 37 =========================================
15.141292638624341

Trial 38 

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 15.143550903716138
Avg = 15.081683090978833
Std = 0.1945146455891316


In [7]:
print(y_max_arr.tolist())

[14.718780135141754, 15.140761744346028, 15.115421216308118, 14.66197904907688, 15.143119125830182, 15.128998035991236, 15.141162558666206, 15.0951820894145, 15.1228548249069, 15.142239516127237, 15.140542559265285, 15.14049544868148, 15.142755956743358, 15.143534109723822, 15.098487926487866, 15.141837609366027, 13.40810633659945, 15.143458702954012, 14.970072580055252, 15.143273847124165, 15.095537657961504, 15.062108790363952, 15.143199520009844, 14.673293612759077, 15.127779455441807, 15.119938754298055, 15.143550903716138, 14.903334712701191, 15.143282161723091, 15.143145666202487, 15.137713342893727, 15.04037609956161, 15.13860060888496, 15.093387013010044, 15.14248177460907, 15.057953924987444, 15.140282624068348, 15.141292638624341, 15.051043417595617, 15.143327626892066, 15.13964470028302, 15.11846447386404, 15.141737005372029, 15.128477427312337, 15.14314702335078, 15.141506844958304, 15.143550192160102, 15.100415089781446, 15.093363136397109, 14.842583797813916, 15.141988464

In [8]:
# filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/SequentialTestswGPModel/DataGenerated/normal_PI_9_27_3.pkl"
# latestdf = pd.DataFrame(y_max_arr)
# pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    15.136415
1    15.077466
2    15.141937
3    15.143064
4    15.138300
..         ...
995  15.136440
996  15.142615
997  15.086474
998  15.135351
999  15.033513

[1000 rows x 1 columns]
